# AHM Movies — Kaggle GPU Worker

This notebook runs the AHM Movies processing worker on a Kaggle GPU session and exposes it through a temporary Cloudflare Tunnel.

**Before running:** In Notebook Settings, turn **Internet ON** and select a GPU accelerator (P100/T4 if available). Start the notebook with a GPU. This notebook deliberately stops if no GPU is detected instead of silently falling back to slow CPU processing. Kaggle GPU quotas and session limits apply; a free GPU is not guaranteed and a session can be interrupted.

Enter credentials privately when prompted. Do not paste keys into notebook code or GitHub.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg fonts-dejavu-core
!pip -q install --upgrade fastapi==0.115.14 uvicorn==0.34.3 boto3==1.38.46 httpx==0.28.1 faster-whisper==1.1.1 edge-tts==7.2.0 python-multipart==0.0.20
!git clone --depth 1 --branch feature/kaggle-worker https://github.com/socialboosthub/ahm-movies.git /kaggle/working/ahm-movies
%cd /kaggle/working/ahm-movies/worker
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No Kaggle GPU detected. Open Notebook Settings, select a GPU accelerator, then restart and run all cells.')
print('GPU ready:', torch.cuda.get_device_name(0))
print('Dependencies installed.')

## 2. Enter private credentials

Use the same Cloudflare R2 bucket credentials and Groq API key already configured for AHM Movies. A new worker token is generated; it must be copied into Vercel as `RECAP_WORKER_TOKEN` together with the tunnel URL as `RECAP_WORKER_URL`.


In [ ]:
import os, getpass, secrets, subprocess, pathlib, time, re, requests, platform, urllib.request
os.environ['R2_ACCOUNT_ID'] = getpass.getpass('Cloudflare R2 Account ID: ')
os.environ['R2_ACCESS_KEY_ID'] = getpass.getpass('R2 Access Key ID: ')
os.environ['R2_SECRET_ACCESS_KEY'] = getpass.getpass('R2 Secret Access Key: ')
os.environ['R2_BUCKET'] = input('R2 bucket name: ').strip()
os.environ['GROQ_API_KEY'] = getpass.getpass('Groq API key: ')
os.environ['LLM_MODEL'] = 'openai/gpt-oss-20b'
os.environ['TRANSCRIBE_PROVIDER'] = 'groq'
os.environ['WHISPER_DEVICE'] = 'cuda'
os.environ['WHISPER_COMPUTE_TYPE'] = 'float16'
os.environ['WHISPER_MODEL'] = 'large-v3-turbo'
os.environ['FFMPEG_PRESET'] = 'veryfast'
os.environ['RECAP_WORKER_TOKEN'] = secrets.token_urlsafe(36)
print('\nCopy this worker token privately into Vercel as RECAP_WORKER_TOKEN:\n')
print(os.environ['RECAP_WORKER_TOKEN'])
print('\nCredentials loaded into this Kaggle session only.')

## 3. Start the worker and public tunnel

Keep this Kaggle session running while a movie is processing. The temporary tunnel URL changes after a new session starts.


In [ ]:
arch = platform.machine().lower()
if arch not in ('x86_64', 'amd64'):
    raise RuntimeError(f'Unsupported Kaggle CPU architecture: {arch}')
urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '/usr/local/bin/cloudflared')
os.chmod('/usr/local/bin/cloudflared', 0o755)
worker_log = open('/kaggle/working/ahm-worker.log', 'w')
worker_proc = subprocess.Popen(['python','-m','uvicorn','movie_api:app','--host','127.0.0.1','--port','8000'], cwd='/kaggle/working/ahm-movies/worker', stdout=worker_log, stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:8000/health', timeout=3).ok: break
    except Exception: pass
    time.sleep(2)
else:
    print(pathlib.Path('/kaggle/working/ahm-worker.log').read_text(errors='ignore')[-6000:])
    raise RuntimeError('Worker did not start. See worker log above.')
tunnel_log = open('/kaggle/working/ahm-tunnel.log', 'w')
tunnel_proc = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT)
worker_url = None
for _ in range(60):
    time.sleep(2)
    tunnel_log.flush()
    log_text = pathlib.Path('/kaggle/working/ahm-tunnel.log').read_text(errors='ignore')
    match = re.search(r'https://[a-zA-Z0-9-]+\\.trycloudflare\\.com', log_text)
    if match:
        worker_url = match.group(0)
        break
if not worker_url:
    raise RuntimeError('Cloudflare Tunnel URL was not found. Check /kaggle/working/ahm-tunnel.log.')
print('AHM Movies worker URL:', worker_url)
print('Health check:', requests.get(worker_url + '/health', timeout=30).json())
print('GPU:', torch.cuda.get_device_name(0))
print('Keep this Kaggle session running while processing.')

## 4. Connect the Vercel Preview deployment

In Vercel → `ahm-movies` → Settings → Environment Variables, set **Preview** for branch `feature/kaggle-worker`:

- `RECAP_WORKER_URL` = the temporary `https://....trycloudflare.com` URL printed above (no trailing slash)
- `RECAP_WORKER_TOKEN` = the generated token from section 2

Keep the existing R2 variables in Vercel. Redeploy the Preview after changing these two variables. Do not change Production yet. First test the Preview with a short clip, then a longer file. A Kaggle session can stop or be reclaimed, and a new session requires a new URL/token and updated Preview variables.


In [ ]:
print('Worker running:', worker_proc.poll() is None)
print('Tunnel running:', tunnel_proc.poll() is None)
print('Health:', requests.get(worker_url + '/health', timeout=20).json())
print('GPU:', torch.cuda.get_device_name(0))